<a href="https://colab.research.google.com/github/RobertSichomba/Commodity_Prices/blob/main/Homework4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [32]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import pandas as pd
import warnings
import seaborn as sns
warnings.filterwarnings("ignore")
from sklearn.covariance import EllipticEnvelope
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn import svm
from sklearn.covariance import EmpiricalCovariance
from sklearn.covariance import MinCovDet
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from IPython.display import display, Image

In [23]:
from google.colab import drive


In [24]:
file_path = "/content/drive/My Drive/InnerMongolia_CompositionalData (1).csv"

def data(file_path: str) -> pd.DataFrame:
    """Load and display basic information of the given dataset."""
    try:
        # Use pd.read_csv for CSV files, not pd.read_excel
        df = pd.read_csv(file_path)
        print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
        print("\nFirst 5 rows:")
        print(df.head())
        return df
    except Exception as e:
        print(f"Error: {e}")
        return None

# Load the dataset
data = data(file_path)


Rows: 968, Columns: 32

First 5 rows:
          x       y      Au        Ag       As     Ba      Be       Bi  \
0  28.09000  28.306  0.8397  0.163580  12.5570  640.0  2.5660  0.41059   
1  21.73600  34.540  0.5309  0.097287   8.7926  610.4  2.4512  0.28667   
2  30.46400  26.078  2.6924  0.119510  14.8070  535.3  2.4506  0.35381   
3  30.61700  27.809  1.4280  0.149940  15.7080  565.0  2.4934  0.27281   
4   0.76984  23.904  0.6721  0.064590   5.4560  521.5  2.2487  0.22260   

        Cd       Co  ...       W      Zn     Zr        Si        Al     Fe  \
0  0.15289  10.2680  ...  1.7011  68.816  321.2  639300.0  142900.0  41400   
1  0.11157   8.9944  ...  1.5100  52.250  311.7  658800.0  122400.0  29800   
2  0.10856  11.7280  ...  1.8328  65.283  254.6  570200.0  131700.0  49100   
3  0.15581   9.3256  ...  1.4129  66.145  269.9  584600.0  135400.0  45900   
4  0.08863   5.5854  ...  1.1380  38.857  529.6  662200.0  124700.0  26900   

        Mg       Ca     Na      K  
0  11700.0  

In [31]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## The is Question 1: CODA EDA.

For this analysis, I will start with a basic CODA Exploratory Data Analysis (EDA). This will include generating a variation array, correlation plots, statistics plots, biplots and ternary diagrams to explore the structure of the dataset.

In [26]:
image_path = "/content/drive/MyDrive/Screenshot (79).png"
display(Image(filename=image_path))

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Screenshot (79).png'

In [27]:
image_path = "/content/drive/MyDrive/Screenshot (80).png" # Check this path
display(Image(filename=image_path))

In [ ]:
image_path = "/content/drive/MyDrive/Screenshot (76).png" # Check this path
display(Image(filename=image_path))

In [ ]:
image_path = "/content/drive/MyDrive/Screenshot (77).png"
display(Image(filename=image_path))

## This Is Question 2: Interpretation of The EDA Findings

In question 1 we conducted an EDA using CODApack. The biplots, ternary plots and variation array using centered log-ratio (clr) transformations gave a lot of insights before proceeding to question 3.

One of the most important observations was the dominant projection of calcium (Ca), whose vector appeared longest and pointed in the opposite direction to Ag.This basically mean that Ca varried the most in our dataset however, uncorrelate to Ag.

From the biplot also, a cluster of elements including Cr, Zn, W, Bi, Mn, and Ag was observed with shorter vectors pointing in a similar direction indicating high correlation between these elements.

Other notable groupings or rather clusters included:

- Ni, Cu, Sb, and Co, partly aligned with Ag, indicating some correlation too.

- Ba, Rb, K, Be, Zr, and Al, which formed a separate cluster.

- A final cluster involving Ti, Pb, Sn, Nb, and P.

These groupings were further supported by the variation array where Ca had the highest variance indicated by the red boxes, some of the important log-ratios I observed are Ag/As and Pb/Sb which confirmed strong geochemical linkages.

## This Is Question 3: Outlier detection on the compositional data by means of robust Mahalanobis distance analysis.

In [ ]:
! pip install scikit-bio --quiet
from skbio.stats.composition import *

In [ ]:
import skbio

In [ ]:
import scipy.stats as stats
from sklearn.covariance import MinCovDet, EmpiricalCovariance
from skbio.stats.composition import clr

In [ ]:
print(skbio.__version__)

In [ ]:
compositional_data = data.values[:,2:]
columns_name = data.columns[2:]
clr_data = clr(compositional_data + 1e-6)
clr_df = pd.DataFrame(clr_data, columns=[f'clr({col})' for col in columns_name])

In [ ]:
clr_df.columns

In [ ]:
X = clr_df[['clr(Ag)', 'clr(Pb)', 'clr(As)', 'clr(Sb)']].values
robust_cov = MinCovDet().fit(X)
RMD = np.sqrt(robust_cov.mahalanobis(X))

dof = X.shape[1]
threshold = np.sqrt(stats.chi2.ppf(0.95, df=dof))
outliers = RMD > threshold

In [ ]:
plt.figure(figsize=(10, 8))
plt.scatter(data['x'], data['y'],
            c='gray', s=30, label='Inliers')
plt.scatter(data['x'][outliers], data['y'][outliers],
            c='red', s=80, edgecolor='k', label='Outliers')
plt.xlabel("x")
plt.ylabel("y")
plt.title("Spatial Distribution of Outliers (Robust Mahalanobis Distance)")
plt.legend()
plt.show()

In [ ]:
# Create the plot
plt.figure(figsize=[6, 4])
plt.scatter(X[:, 0], X[:, 1], c=outlier_index, s=40, edgecolors='k', cmap='Greys')

# Define center and radius based on data
center = (np.mean(X[:, 0]), np.mean(X[:, 1]))
radius = np.std(np.sqrt((X[:, 0] - center[0])**2 + (X[:, 1] - center[1])**2))  # More consistent radius

# Plot the circular demarcation
circle = plt.Circle(center, radius, color='k', fill=False, linewidth=2)
plt.gca().add_patch(circle)  # Use add_patch instead of add_artist for clarity

# Labels and title
plt.xlabel('clr(Ag)')
plt.ylabel('clr(Pb)')
plt.title('Robust Mahalanobis Distance (Approx. Circular Boundary)')
plt.axis('equal')  # Optional: keep x and y scales consistent
plt.show()


In [30]:
# Create the plot
plt.figure(figsize=[6, 4])
plt.scatter(X[:, 0], X[:, 1], c=outlier_index, s=40, edgecolors='k', cmap='Greys')

# Define center and slightly larger radius
center = (np.mean(X[:, 0]), np.mean(X[:, 1]))
radius = np.std(np.sqrt((X[:, 0] - center[0])**2 + (X[:, 1] - center[1])**2)) * 2.5  # Increased radius

# Plot the wider circle
circle = plt.Circle(center, radius, color='k', fill=False, linewidth=2)
plt.gca().add_patch(circle)

# Labels and title
plt.xlabel('clr(Ag)')
plt.ylabel('clr(Pb)')
plt.title('Robust Mahalanobis Distance (Wider Boundary)')
plt.axis('equal')
plt.show()


NameError: name 'outlier_index' is not defined

<Figure size 600x400 with 0 Axes>

Using robust Mahalanobis distance analysis on clr-transformed compositional data (Ag, Pb, As, Sb), several samples were identified as multivariate outliers. These points exhibit statistically significant deviation from the central data structure and may indicate unusual geochemical signatures. Spatial plots reveal clustering of these outliers in particular regions, suggesting localized geogenic anomalies or potential contamination hotspots. The use of a robust estimator ensured that the presence of existing outliers did not distort the covariance structure, leading to a more reliable detection of multivariate anomalies.

## This is Question 4: Clustering Analysis

Clustering is an unsupervised machine learning technique that aims to group similar instances into distinct clusters based on their feature similarity. The goal is to group similar instances together into clusters.

Applied on geochemical data this algorithm helps to identify mineralized zones, detect geochemical anomalies, and guide exploration efforts "As we all know drilling is expensive!". This analysis applies these clustering techniques to geochemical data from Northeastern Inner Mongolia to uncover spatial and compositional patterns associated with mineralization.

This is homework two just before concluding the chapter with homework five. The lectures, including research on these protocals, and applying them on a real dataset is amazing experience!

## K-Means

K-Means is one of the simplest and most widely used clustering algorithms in unsupervised learning. It performs especially well on datasets with spherical clusters that are relatively uniform in size and density — like the classic scenario of five distinguishable blobs often used in examples.

The algorithm partitions the dataset into a predefined number of clusters (K) by minimizing the within cluster sum of squares (inertia). It does this iteratively, updating cluster centroids and reassigning data points until convergence.

In [ ]:
import os
import pandas as pd
from skbio.stats.composition import clr

# Option 1: Change working directory (if you know where the file is)
# os.chdir('/path/to/directory/containing/file')  # Replace with the actual path

# Option 2: Provide the full path to the file
# file_path = '/path/to/geochemical_data.csv'  # Replace with the actual path
# Instead of a placeholder, use the actual file_path from your environment.
file_path = "/content/drive/My Drive/InnerMongolia_CompositionalData (1).csv"

# Load data and select elements
data = pd.read_csv(file_path, index_col=0)  # Using file_path variable now
elements = ['Ag', 'Pb', 'As', 'Sb', 'Zn', 'Cu']
subset = data[elements]

# CLR transformation
subset_clr = clr(subset + 1e-6)  # Handle zeros

In [ ]:
import numpy as np

# Compute pairwise Aitchison distances
aitchison_dist = np.sqrt(np.sum((subset_clr[:, None] - subset_clr) ** 2, axis=2))

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage

Z = linkage(aitchison_dist, method='ward')
plt.figure(figsize=(12, 5))
dendrogram(Z, labels=data.index)
plt.title('Hierarchical Clustering Dendrogram')
plt.xlabel('Sample Index')
plt.ylabel('Aitchison Distance')
plt.show()

In [ ]:
from sklearn.manifold import MDS

mds = MDS(n_components=2, dissimilarity='precomputed', random_state=42)
mds_coords = mds.fit_transform(aitchison_dist)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, elem in enumerate(elements):
    scatter = axes[i].scatter(mds_coords[:, 0], mds_coords[:, 1],
                             c=subset[elem], cmap='viridis', s=50, edgecolor='k')
    plt.colorbar(scatter, ax=axes[i], label=f'{elem} (ppm)')
    axes[i].set_title(f'MDS Colored by {elem}')

plt.tight_layout()
plt.show()

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from skbio.stats.composition import clr
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.manifold import MDS
from sklearn.cluster import KMeans

# Step 1: Load the data
data = pd.read_csv("geochemical_data.csv", index_col=0)

# Step 2: Select the top six elements based on CODA EDA
elements = ['Ag', 'Pb', 'As', 'Sb', 'Zn', 'Cu']  # Top six elements
subset = data[elements]

# Step 3: CLR Transformation
# Add a small constant to avoid log(0) issues
subset_clr = clr(subset + 1e-6)  # Applying CLR transformation

# Step 4: Aitchison Distance Matrix
# Compute pairwise Aitchison distances
def aitchison_distance(X):
    """
    Compute pairwise Aitchison distances.
    :param X: CLR-transformed data
    :return: Aitchison distance matrix
    """
    return np.sqrt(np.sum((X[:, None] - X) ** 2, axis=2))

aitchison_dist = aitchison_distance(subset_clr)

# Step 5: K-means Clustering
# Perform K-means clustering on CLR-transformed data
kmeans = KMeans(n_clusters=2, random_state=42)  # Choose number of clusters
kmeans_labels = kmeans.fit_predict(subset_clr)

# Step 6: Multidimensional Scaling (MDS)
mds = MDS(n_components=2, dissimilarity='precomputed', random_state=42)
mds_coords = mds.fit_transform(aitchison_dist)

# Step 7: Generate MDS plots colored by K-means clustering labels
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, elem in enumerate(elements):
    scatter = axes[i].scatter(mds_coords[:, 0], mds_coords[:, 1], c=subset[elem], cmap='viridis', s=50, edgecolor='k')
    plt.colorbar(scatter, ax=axes[i], label=f'{elem} (ppm)')
    axes[i].set_title(f'MDS Colored by {elem}')

plt.tight_layout()
plt.show()

# Step 8: MDS Plot Colored by K-means Clustering
plt.figure(figsize=(8, 6))
plt.scatter(mds_coords[:, 0], mds_coords[:, 1], c=kmeans_labels, cmap='viridis', s=50, edgecolor='k')
plt.title('MDS Plot Colored by K-means Clustering')
plt.xlabel('MDS Component 1')
plt.ylabel('MDS Component 2')
plt.colorbar(label='Cluster')
plt.show()



In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from skbio.stats.composition import clr
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.manifold import MDS
from sklearn.cluster import KMeans

# Step 1: Load the data
# Instead of 'geochemical_data.csv', use the path to your actual data file.
# Assuming 'InnerMongolia_CompositionalData (1).csv' is the correct file
data = pd.read_csv("/content/drive/My Drive/InnerMongolia_CompositionalData (1).csv", index_col=0)

# Step 2: Select the top six elements based on CODA EDA
elements = ['Ag', 'Pb', 'As', 'Sb', 'Zn', 'Cu']  # Top six elements
subset = data[elements]

# Step 3: CLR Transformation
# Add a small constant to avoid log(0) issues
subset_clr = clr(subset + 1e-6)  # Applying CLR transformation

# Step 4: Aitchison Distance Matrix
# Compute pairwise Aitchison distances
def aitchison_distance(X):
    """
    Compute pairwise Aitchison distances.
    :param X: CLR-transformed data
    :return: Aitchison distance matrix
    """
    return np.sqrt(np.sum((X[:, None] - X) ** 2, axis=2))

aitchison_dist = aitchison_distance(subset_clr)

# Step 5: K-means Clustering
# Perform K-means clustering on CLR-transformed data
kmeans = KMeans(n_clusters=3, random_state=42)  # Choose number of clusters
kmeans_labels = kmeans.fit_predict(subset_clr)

# Step 6: Multidimensional Scaling (MDS)
mds = MDS(n_components=2, dissimilarity='precomputed', random_state=42)
mds_coords = mds.fit_transform(aitchison_dist)

# Step 7: Generate MDS plots colored by K-means clustering labels
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, elem in enumerate(elements):
    scatter = axes[i].scatter(mds_coords[:, 0], mds_coords[:, 1], c=subset[elem], cmap='viridis', s=50, edgecolor='k')
    plt.colorbar(scatter, ax=axes[i], label=f'{elem} (ppm)')
    axes[i].set_title(f'MDS Colored by {elem}')

plt.tight_layout()
plt.show()

# Step 8: MDS Plot Colored by K-means Clustering
plt.figure(figsize=(8, 6))
plt.scatter(mds_coords[:, 0], mds_coords[:, 1], c=kmeans_labels, cmap='viridis', s=50, edgecolor='k')
plt.title('MDS Plot Colored by K-means Clustering')
plt.xlabel('MDS Component 1')
plt.ylabel('MDS Component 2')
plt.colorbar(label='Cluster')
plt.show()

In [ ]:
 fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize = (18,8))

plot1 = ax1.scatter(data['x'],data['y'],c = data['Ag'],cmap = 'Spectral_r',vmin = 0.04, vmax = 0.15)
plt.colorbar(plot1,ax=ax1)
ax1.set_xlabel('x')
ax1.set_ylabel('y')
ax1.set_title('Ag')
plot2 = ax2.scatter(data['x'],data['y'],c = data['Au'],cmap = 'Spectral_r',vmin = 0, vmax = 2)
plt.colorbar(plot2,ax=ax2)
ax2.set_xlabel('x')
ax2.set_ylabel('y')
ax2.set_title('Au')
plot3 =ax3.scatter(data['x'],data['y'],c = data['Cu'],cmap = 'Spectral_r',vmin = 0, vmax = 40)
plt.colorbar(plot3,ax=ax3)
ax3.set_xlabel('x')
ax3.set_ylabel('y')
ax3.set_title('Cu')

## additive log-ratio transformation on compositional data

You need to choose the base element for alr transformation: log(components/base)

In [ ]:
def alr(data, columns_name, choice_D):
    # choice D: default = , the last component
    choice_D = np.where(columns_name == choice_D)[0][0]
    alr_result = np.log(data/data[:,choice_D].reshape(-1,1))
    name_D = columns_name[choice_D]
    columns_name = np.delete(columns_name,choice_D,0)
    alr_result = pd.DataFrame(data = np.delete(alr_result, choice_D, 1), columns = [('log(')+str(name)+'/'+str(name_D)+')' for name in columns_name])
    return alr_result

In [ ]:
compositional_data = data.values[:,2:]
columns_name = data.columns[2:]

In [ ]:
## alr transformation

######################################################################
#Choose the base from EDA, you should determine your base by yourself#
######################################################################
base = 'Ca'
alr_data = alr(compositional_data, columns_name, choice_D = base)

In [ ]:
alr_data

## This is Question 2:

# Principal Component Analysis



The PCA protocol:
- Make scatterplots, make biplots, perform EDA
- Run PCA code
- Make scree plot, score plots to investigate variance contributions
- Investigate PC vectors or “loadings”, can you find patterns
- Make maps of PC scores on space, do you see spatial correlation

In [ ]:
from sklearn.decomposition import PCA
X = alr_data.values
pca = PCA(n_components=X.shape[1])
pca.fit(X)
X_pc = pca.transform(X)

## Scree plot

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(np.arange(1,X.shape[1]+1), pca.explained_variance_ratio_.cumsum(), \
          marker='o', markersize=5, linestyle = 'dashed', color='blue', label='Cumulative')
plt.plot(np.arange(1,X.shape[1]+1), pca.explained_variance_ratio_, \
          marker='o', markersize=5, color='red', label='Variance proportion')
plt.xlabel('number of PCs', fontsize = 13)
plt.ylabel('proportion of variance explained', fontsize = 13)
plt.xticks(np.arange(1,X.shape[1]+1,1))
plt.ylim(0,1.1)
plt.legend()
plt.title('Scree plot', fontsize = 14)
plt.grid(linestyle='dashed')

## Score plots

Colored by the original compositions

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# Assuming 'alr_data' is your preprocessed data
X = alr_data.values
pca = PCA(n_components=X.shape[1])
pca.fit(X)
X_pc = pca.transform(X)  # Calculate X_pc here

plt.figure(figsize=(10,6))
color_name = 'Ag' # you can change the colored composition here

plt.scatter(X_pc[:,0], X_pc[:,1],  c=data[color_name],
            cmap = 'Reds', edgecolor='k', linewidth=0.2)

plt.colorbar(fraction=0.02)
plt.ylabel('PC2 scores ({:.1f}% var)'.format(100*pca.explained_variance_ratio_[1]), fontsize=12)
plt.xlabel('PC1 scores ({:.1f}% var)'.format(100*pca.explained_variance_ratio_[0]), fontsize=12)

plt.title('PC score plots, colored by '+color_name, fontsize=12)

Colored by the log-ratios

In [ ]:
plt.figure(figsize=(10,6))
color_name = ['Ag','Ba'] # you can change the colored composition here,

plt.scatter(X_pc[:,0], X_pc[:,1],  c=np.log(data[color_name[0]]/data[color_name[1]]),
            cmap = 'Reds', edgecolor='k', linewidth=0.2)

plt.colorbar(fraction=0.02)
plt.ylabel('PC2 scores ({:.1f}% var)'.format(100*pca.explained_variance_ratio_[1]), fontsize=12)
plt.xlabel('PC1 scores ({:.1f}% var)'.format(100*pca.explained_variance_ratio_[0]), fontsize=12)

plt.title('PC score plots, colored by log('+color_name[0]+'/'+color_name[1]+')', fontsize=12)

## Principal Component (PC) vectors/loadings

In [ ]:
plt.figure(figsize=(30,6))
for i in range(4):
  plt.subplot(2,2,i+1)
  plt.bar(alr_data.columns,
          pca.components_[i], edgecolor='k', linewidth=0.5, width=0.6)
  plt.hlines(y=0, xmin=-1, xmax=30 )
  plt.ylabel('PC loadings', fontsize=13)
  plt.xticks(fontsize=11,rotation='vertical')
  plt.title('PC'+str(i+1)+'({:.1f}% var)'.format(100*pca.explained_variance_ratio_[i]))
  plt.xlim(-0.6,29.6)
  plt.ylim(-1,1)
plt.tight_layout()


## PC scores on the map

In [ ]:
# you can change the colorbar limits using vmin and vmax
PC_idx = 2 #here PC_idx 3 visualize the PC3 score on the map. you can change PC_idx to plot other PC scores
plt.figure(figsize=(4.5,6))
plt.scatter(data['x'],data['y'],
            c = X_pc[:,PC_idx-1], cmap = 'RdBu_r', vmax=2, vmin=-2, edgecolor='k', linewidth=0.1)
plt.colorbar(fraction=0.03)
plt.title('PC'+str(PC_idx)+' score ({:.1f}% var)'.format(100*pca.explained_variance_ratio_[PC_idx-1]))

plt.xlabel('x')
plt.ylabel('y')
plt.tight_layout()

# Factor Analysis

The FA protocol:
- Think first about the problem itself: how many factors do you hypothesize?
- Perform factor analysis with the hypothesized # factors
- Analyze the factor analysis table to assess the fit: what is common vs what is error
- Plot loadings
- Project data on factor vectors to get factor scores
- Plot scores into a map

In [ ]:
! pip install factor_analyzer --quiet

In [ ]:
X = alr_data.values

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaler = scaler.fit_transform(X)

from factor_analyzer import FactorAnalyzer
# Create factor analysis object and perform factor analysis
fa = FactorAnalyzer(n_factors = 3, rotation = 'varimax',method='minres')
fa.fit(X)
X_fa =fa.transform(X)

In [ ]:
## explained variance for each factor
commumalities = np.sum(fa.loadings_.T**2, axis=1)
var_explained_prop = commumalities/X.shape[1]

## Analyze the factor analysis table to assess the fit

In [ ]:
pd.DataFrame(np.round(np.corrcoef(X.T),2), columns = alr_data.columns, index = alr_data.columns)

### Common

In [ ]:
pd.DataFrame(np.round(np.matmul(fa.loadings_,fa.loadings_.T),2), columns = 'Comm_'+alr_data.columns, index = 'Comm_'+alr_data.columns)

### Error

Theoretical error

In [ ]:
pd.DataFrame(np.round(np.diag(fa.get_uniquenesses()),2), columns = 'Error_'+alr_data.columns, index = 'Error_'+alr_data.columns)

Empirical error

In [ ]:
epsilon = X_scaler-np.matmul(X_fa,fa.loadings_.T)

In [ ]:
pd.DataFrame(np.round(np.cov(epsilon.T),2), columns = 'Error_'+alr_data.columns, index = 'Error_'+alr_data.columns)

## Factor loadings

In [ ]:
import sys
sys.path.append('/content/DataScienceForGeosciences/hw4/')
from loadings_plot import *

In [ ]:
plt.figure(figsize=(30,6))
for i in range(3):
  plt.subplot(2,2,i+1)
  plt.bar(alr_data.columns,
          fa.loadings_.T[i], edgecolor='k', linewidth=0.5, width=0.6)
  plt.hlines(y=0, xmin=-1, xmax=30 )
  plt.ylabel('Factor loadings', fontsize=13)
  plt.xticks(fontsize=11,rotation=45)
  plt.title('Factor '+str(i+1)+'({:.1f}% var)'.format(100*fa.get_factor_variance()[1][i]))
  plt.xlim(-0.6,29.6)
  plt.ylim(-1.05,1.05)
plt.tight_layout()

In [ ]:
columns_name = data.columns[2:]
columns_name = np.delete(columns_name,np.where(columns_name == base)[0][0],0)

In [ ]:
loadings_plot(fa.loadings_.T, list(columns_name), var_explained_prop, n_fctr=3) ## should be log(component/base), we just plot the component for better visualization

## Factor scores on the map

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize = (18,8))
# you can change the colorbar limits using vmin and vmax

plot1 = ax1.scatter(data['x'],data['y'],c = X_fa[:,0],cmap = 'RdBu_r',vmin = -3, vmax = 3)
plt.colorbar(plot1,ax=ax1)
ax1.set_xlabel('Longtitude')
ax1.set_ylabel('Longtitude')
ax1.set_title('F1')
plot2 = ax2.scatter(data['x'],data['y'],c = X_fa[:,1],cmap = 'RdBu_r',vmin = -3, vmax = 3)
plt.colorbar(plot2,ax=ax2)
ax2.set_xlabel('Longtitude')
ax2.set_ylabel('Longtitude')
ax2.set_title('F2')
plot3 =ax3.scatter(data['x'],data['y'],c = X_fa[:,2],cmap = 'RdBu_r',vmin = -3, vmax = 3)
plt.colorbar(plot3,ax=ax3)
ax3.set_xlabel('x')
ax3.set_ylabel('y')
ax3.set_title('F3')